In [1]:
# Kill all processes on the GPU
!fuser -v /dev/nvidia* -k

In [2]:
# Check GPU status
!nvidia-smi

Tue Apr 21 16:30:03 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   48C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [3]:
# For Qwen2.5-VL
# # %%capture
# import os, re
# if "COLAB_" not in "".join(os.environ.keys()):
#     %pip install unsloth  # Do this in local & cloud setups
# else:
#     import torch; v = re.match(r'[\d]{1,}\.[\d]{1,}', str(torch.__version__)).group(0)
#     xformers = 'xformers==' + {'2.10':'0.0.34','2.9':'0.0.33.post1','2.8':'0.0.32.post2'}.get(v, "0.0.34")
#     %pip install sentencepiece protobuf "datasets==4.3.0" "huggingface_hub>=0.34.0" hf_transfer
#     %pip install --no-deps unsloth_zoo bitsandbytes accelerate {xformers} peft trl triton unsloth
# %pip install transformers==4.56.2
# %pip install --no-deps trl==0.22.2

# For Qwen3-VL
# %%capture
import os, re
if "COLAB_" not in "".join(os.environ.keys()):
    %pip install unsloth  # Do this in local & cloud setups
else:
    import torch; v = re.match(r'[\d]{1,}\.[\d]{1,}', str(torch.__version__)).group(0)
    xformers = 'xformers==' + {'2.10':'0.0.34','2.9':'0.0.33.post1','2.8':'0.0.32.post2'}.get(v, "0.0.34")
    %pip install sentencepiece protobuf "datasets==4.3.0" "huggingface_hub>=0.34.0" hf_transfer
    %pip install --no-deps unsloth_zoo bitsandbytes accelerate {xformers} peft trl triton unsloth
%pip install transformers==4.57.1
%pip install --no-deps trl==0.22.2

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 35.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 138.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 21.7 MB/s eta 0:00:00:00:0100:01
  Attempting uninstall: pyarrow
    Found existing installation: pyarrow 18.1.0
    Uninstalling pyarrow-18.1.0:
      Successfully uninstalled pyarrow-18.1.0
  Attempting uninstall: datasets
    Found existing installation: datasets 4.0.0
    Uninstalling datasets-4.0.0:
      Successfully uninstalled datasets-4.0.0
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 55.8/55.8 kB 7.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 110.8/110.8 MB 8.4 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 421.2/421.2 kB 29.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 14.0 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 697.4/697.4 kB 62.0 MB/s eta 0:00:00
   ━━━━━━

In [4]:
# # For Kaggle setup
# %pip install unsloth
# %pip install --upgrade "torchvision>=0.25.0"

# Configuration

In [5]:
SEED = 42

# Model configuration
# MODEL_ID = 'unsloth/Qwen2.5-VL-7B-Instruct-bnb-4bit'
# MODEL_ID = 'unsloth/Qwen3-VL-8B-Instruct-unsloth-bnb-4bit'
# MODEL_ID = 'unsloth/Qwen3-VL-8B-Instruct'
MODEL_ID = 'unsloth/Qwen3-VL-8B-Thinking'

# Visual codebook configuration
K = 8192

BVIR_DATA_ID = 'alxxtexxr/BVIR-Data'
# CODEBOOK_PATH = 'model_unsloth_Qwen2.5-VL-7B-Instruct-bnb-4bit.data_sft_jxie_coco_captions_1000.data_rl_alxxtexxr_ViRL1.25K_1000/codebook_faiss_8192.npy'
# CODEBOOK_PATH = 'model_unsloth_Qwen3-VL-8B-Instruct-unsloth-bnb-4bit_fp16.data_sft_jxie_coco_captions_1000.data_rl_alxxtexxr_ViRL1.25K_1000/codebook_faiss_8192.npy'
# CODEBOOK_PATH = 'model_unsloth_Qwen3-VL-8B-Instruct_fp16.data_sft_jxie_coco_captions_1000.data_rl_alxxtexxr_ViRL1.25K_1000/codebook_faiss_8192.npy'
CODEBOOK_PATH = 'model_unsloth_Qwen3-VL-8B-Thinking_fp16.data_sft_jxie_coco_captions_1000.data_rl_alxxtexxr_ViRL1.25K_1000/codebook_faiss_8192.npy'

In [6]:
import os
from unsloth import FastVisionModel
import random
import numpy as np
import torch
import transformers

def set_seed(seed, verbose=True):
    # Set random seed for os
    os.environ['PYTHONHASHSEED'] = str(seed)
    os.environ['CUBLAS_WORKSPACE_CONFIG'] = ':16:8'  # CUDA deterministic behavior (11.2+)

    # Set random seed for NumPy
    np.random.seed(seed)

    # Set random seed for Torch
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)           # If using multi-GPU
    # torch.use_deterministic_algorithms(True) # ERROR!
    torch.backends.cudnn.deterministic = True  # Ensures deterministic results
    torch.backends.cudnn.benchmark = False     # Avoids non-deterministic algorithms

    # Set random seed for Transformers
    transformers.set_seed(seed)

    # Set random seed for sklearn and Python's own random module
    random.seed(seed) 
    
    if verbose:
        print(f"Random seed: {seed}")

set_seed(SEED)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
Random seed: 42


In [7]:
# Fix TorchDynamo issue
os.environ['TORCHDYNAMO_DISABLE'] = '1'
os.environ['UNSLOTH_DISABLE_COMPILE'] = '1'
os.environ['UNSLOTH_DISABLE_FUSED_KERNELS'] = '1'

# Fix Unsloth issue
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = ''

# Model

In [8]:
import copy

model, tokenizer = FastVisionModel.from_pretrained(
    model_name = MODEL_ID,
    load_in_4bit = False, # Use 4bit to reduce memory use. False for 16-bit LoRA.
    use_gradient_checkpointing = 'unsloth', # True or 'unsloth' for long context
    dtype = torch.float16, # Force FP16
)
tokenizer_orig = copy.deepcopy(tokenizer) # Copy the original tokenizer for comparison later

print(model)

==((====))==  Unsloth 2026.4.6: Fast Qwen3_Vl patching. Transformers: 4.57.1.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.34. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Unsloth: QLoRA and full finetuning all not selected. Switching to 16bit LoRA.


model.safetensors.index.json: 0.00B [00:00, ?B/s]

model-00001-of-00004.safetensors:   0%|          | 0.00/4.90G [00:00<?, ?B/s]

model-00002-of-00004.safetensors:   0%|          | 0.00/4.92G [00:00<?, ?B/s]

model-00003-of-00004.safetensors:   0%|          | 0.00/5.00G [00:00<?, ?B/s]

...

added_tokens.json:   0%|          | 0.00/707 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/614 [00:00<?, ?B/s]

chat_template.jinja: 0.00B [00:00, ?B/s]

video_preprocessor_config.json:   0%|          | 0.00/817 [00:00<?, ?B/s]

Qwen3VLForConditionalGeneration(
  (model): Qwen3VLModel(
    (visual): Qwen3VLVisionModel(
      (patch_embed): Qwen3VLVisionPatchEmbed(
        (proj): Conv3d(3, 1152, kernel_size=(2, 16, 16), stride=(2, 16, 16))
      )
      (pos_embed): Embedding(2304, 1152)
      (rotary_pos_emb): Qwen3VLVisionRotaryEmbedding()
      (blocks): ModuleList(
        (0-26): 27 x Qwen3VLVisionBlock(
          (norm1): LayerNorm((1152,), eps=1e-06, elementwise_affine=True)
          (norm2): LayerNorm((1152,), eps=1e-06, elementwise_affine=True)
          (attn): Qwen3VLVisionAttention(
            (qkv): Linear(in_features=1152, out_features=3456, bias=True)
            (proj): Linear(in_features=1152, out_features=1152, bias=True)
          )
          (mlp): Qwen3VLVisionMLP(
            (linear_fc1): Linear(in_features=1152, out_features=4304, bias=True)
            (linear_fc2): Linear(in_features=4304, out_features=1152, bias=True)
            (act_fn): GELUTanh()
          )
        )
      )
 

In [9]:
# Ensure the embeddings are in float16, not bfloat16, 
# to make it more universal for different types of GPUs
print(model.model.language_model.embed_tokens.weight.dtype)
print(model.lm_head.weight.dtype)
print(model.device)

torch.float16
torch.float16
cuda:0


In [10]:
# Calculate LM embedding norms to set codebook scaling
E = model.get_input_embeddings().weight.data.clone()
lm_norms = E.norm(dim=1)
print("Embedding norms (mean, std):", lm_norms.mean().item(), lm_norms.std().item())

# Randomly sample K norms with replacement
target_norms = lm_norms[torch.randint(0, lm_norms.size(0), (K,))]
print("Target norms (mean, std):", target_norms.mean().item(), target_norms.std().item())

Embedding norms (mean, std): 1.3203125 0.300048828125
Target norms (mean, std): 1.318359375 0.302001953125


In [11]:
def check_embs_info():
    print("Input embeddings shape:", model.get_input_embeddings().weight.shape)
    print("Output embeddings shape:", model.get_output_embeddings().weight.shape)
    print("Weight-tied:", model.get_input_embeddings().weight is model.get_output_embeddings().weight)

check_embs_info()

Input embeddings shape: torch.Size([151936, 4096])
Output embeddings shape: torch.Size([151936, 4096])
Weight-tied: False


In [12]:
list(tokenizer.tokenizer.get_added_vocab().items())[:30]

[('<|endoftext|>', 151643),
 ('<|im_start|>', 151644),
 ('<|im_end|>', 151645),
 ('<|object_ref_start|>', 151646),
 ('<|object_ref_end|>', 151647),
 ('<|box_start|>', 151648),
 ('<|box_end|>', 151649),
 ('<|quad_start|>', 151650),
 ('<|quad_end|>', 151651),
 ('<|vision_start|>', 151652),
 ('<|vision_end|>', 151653),
 ('<|vision_pad|>', 151654),
 ('<|image_pad|>', 151655),
 ('<|video_pad|>', 151656),
 ('<tool_call>', 151657),
 ('</tool_call>', 151658),
 ('<|fim_prefix|>', 151659),
 ('<|fim_middle|>', 151660),
 ('<|fim_suffix|>', 151661),
 ('<|fim_pad|>', 151662),
 ('<|repo_name|>', 151663),
 ('<|file_sep|>', 151664),
 ('<tool_response>', 151665),
 ('</tool_response>', 151666),
 ('<think>', 151667),
 ('</think>', 151668)]

In [13]:
def check_emb_stats(i):
    inner_model = getattr(model, 'model', model)
    x = inner_model.language_model.embed_tokens.weight[i].cpu().detach().numpy()
    print("i:", i)
    print(f"mean: {x.mean()}, max: {x.max()}, min: {x.min()}")
    print("First values:", x[:5])
    print()

check_emb_stats(0) # Check the stats of the first token embedding
check_emb_stats(151664) # Check the stats of the last added token embedding

i: 0
mean: -0.00018262863159179688, max: 0.1474609375, min: -0.1591796875
First values: [ 0.04614  0.02808  0.02478 -0.01636 -0.01843]

i: 151664
mean: 3.9696693420410156e-05, max: 0.035400390625, min: -0.0361328125
First values: [-0.007416  0.004974 -0.002106  0.00787  -0.00586 ]



In [14]:
vthink_tokens = ['<vthink>', '</vthink>']
vtok_tokens = [f'<|vtok_{i}|>' for i in range(K)]
tokenizer.tokenizer.add_tokens(vthink_tokens + vtok_tokens);

In [ ]:
(
    think_start_id, think_end_id, 
    vthink_start_id, vthink_end_id, 
    vtok_start_id, vtok_end_id,
) = tokenizer.tokenizer.encode(
    f'<think></think>'
    f'<vthink></vthink>'
    f'<|vtok_0|><|vtok_{K-1}|>'
)

print("Thinking token start ID:", think_start_id)
print("Thinking token end ID:", think_end_id)
print("Visual thinking token start ID:", vthink_start_id)
print("Visual thinking token end ID:", vthink_end_id)
print("Visual token start ID:", vtok_start_id)
print("Visual token end ID:", vtok_end_id)

Thinking token start ID: 151667
Thinking token end ID: 151668
Visual thinking token start ID: 151669
Visual thinking token end ID: 151670
Visual token start ID: 151671
Visual token end ID: 159862


In [16]:
model.resize_token_embeddings(vtok_end_id+1, mean_resizing=False)
print(model)

OutOfMemoryError: CUDA out of memory. Tried to allocate 1.22 GiB. GPU 0 has a total capacity of 14.56 GiB of which 217.81 MiB is free. Including non-PyTorch memory, this process has 14.35 GiB memory in use. Of the allocated memory 14.18 GiB is allocated by PyTorch, and 17.65 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)

In [ ]:
# model = model.to("cuda")
# torch.cuda.empty_cache()
# # TODO: Clear the CPU memory too

In [18]:
check_emb_stats(0) # Check the stats of the first token embedding
check_emb_stats(151664) # Check the stats of the last added token embedding
check_emb_stats(vtok_start_id) # Check the stats of the first virtual token embedding
check_emb_stats(vtok_end_id) # Check the stats of the last virtual token embedding

i: 0
mean: -0.00018262863159179688, max: 0.1474609375, min: -0.1591796875
First values: [ 0.04614  0.02808  0.02478 -0.01636 -0.01843]

i: 151664
mean: 3.9696693420410156e-05, max: 0.035400390625, min: -0.0361328125
First values: [-0.007416  0.004974 -0.002106  0.00787  -0.00586 ]

i: 151669
mean: -9.012222290039062e-05, max: 0.027587890625, min: -0.027587890625
First values: [-0.0005302 -0.001907  -0.002686  -0.001572  -0.003098 ]

i: 159860
mean: -3.635883331298828e-05, max: 0.07379150390625, min: -0.0738525390625
First values: [-0.02147   0.005352 -0.034     0.04623  -0.01245 ]



In [19]:
from huggingface_hub import hf_hub_download

codebook_path = hf_hub_download(
    repo_id=BVIR_DATA_ID,
    filename=CODEBOOK_PATH,
    repo_type='dataset',
)
print("Visual codebook downloaded to:", codebook_path)

Visual codebook downloaded to: /root/.cache/huggingface/hub/datasets--alxxtexxr--BVIR-Data/snapshots/3090ff44ba44b19e4a40d46eb76cdc436600658e/model_unsloth_Qwen3-VL-8B-Thinking_fp16.data_sft_jxie_coco_captions_1000.data_rl_alxxtexxr_ViRL1.25K_1000/codebook_faiss_8192.npy


In [20]:
import numpy as np

codebook = torch.from_numpy(np.load(codebook_path)).to(torch.float16).to(model.device) # (K, 3584) -> Current: (8192, 3584)
assert codebook.shape[0] == K, f"Expected codebook with {K} entries, but got {codebook.shape[0]}"

print(f"Visual codebook shape: {codebook.shape}, dtype: {codebook.dtype}, device: {codebook.device}")
print("Visual codebook norms before scaling (mean, std):", codebook.norm(dim=1).mean().item(), codebook.norm(dim=1).std().item())

Visual codebook shape: torch.Size([8192, 4096]), dtype: torch.float16, device: cuda:0
Visual codebook norms before scaling (mean, std): 0.79150390625 0.070068359375


In [21]:
# Normalize codebook to unit vectors first
codebook_norm = codebook / codebook.norm(dim=1, keepdim=True) 
print("Visual codebook norms after normalization (mean, std):", codebook_norm.norm(dim=1).mean().item(), codebook_norm.norm(dim=1).std().item())

# Scale to LM-like norms
codebook_scaled = codebook_norm * target_norms[:, None]              
print("Visual codebook norms after scaling (mean, std):", codebook_scaled.norm(dim=1).mean().item(), codebook_scaled.norm(dim=1).std().item())

Visual codebook norms after normalization (mean, std): 1.0 0.00014889240264892578
Visual codebook norms after scaling (mean, std): 1.318359375 0.302001953125


In [22]:
chunk_size = 512

with torch.no_grad():
    for i in range(0, vtok_end_id - vtok_start_id + 1, chunk_size):
        a_start = vtok_start_id + i
        a_end = min(a_start + chunk_size, vtok_end_id + 1)

        b_start = i
        b_end = b_start + (a_end - a_start)
        
        print(f"Processing chunk: codebook_scaled[{b_start}:{b_end}] => embed_tokens[{a_start}:{a_end}]")

        # Move only a chunk to GPU
        chunk = codebook_scaled[b_start:b_end].to(
            model.lm_head.weight.device,
            dtype=model.lm_head.weight.dtype,
            non_blocking=True
        )

        # Replace in the input embeddings
        getattr(model, 'model', model).language_model.embed_tokens.weight[a_start:a_end] = chunk
        
        # Replace in the output embeddings too even if they are not tied, to give better initialization
        model.lm_head.weight[a_start:a_end] = chunk

        # Optional: free reference quickly
        del chunk

Processing chunk: codebook_scaled[0:512] => embed_tokens[151669:152181]
Processing chunk: codebook_scaled[512:1024] => embed_tokens[152181:152693]
Processing chunk: codebook_scaled[1024:1536] => embed_tokens[152693:153205]
Processing chunk: codebook_scaled[1536:2048] => embed_tokens[153205:153717]
Processing chunk: codebook_scaled[2048:2560] => embed_tokens[153717:154229]
Processing chunk: codebook_scaled[2560:3072] => embed_tokens[154229:154741]
Processing chunk: codebook_scaled[3072:3584] => embed_tokens[154741:155253]
Processing chunk: codebook_scaled[3584:4096] => embed_tokens[155253:155765]
Processing chunk: codebook_scaled[4096:4608] => embed_tokens[155765:156277]
Processing chunk: codebook_scaled[4608:5120] => embed_tokens[156277:156789]
Processing chunk: codebook_scaled[5120:5632] => embed_tokens[156789:157301]
Processing chunk: codebook_scaled[5632:6144] => embed_tokens[157301:157813]
Processing chunk: codebook_scaled[6144:6656] => embed_tokens[157813:158325]
Processing chunk:

In [23]:
model.get_input_embeddings().weight is model.get_output_embeddings().weight

False

In [24]:
check_emb_stats(0) # Check the stats of the first token embedding
check_emb_stats(151664) # Check the stats of the last added token embedding
check_emb_stats(152063) # Check the stats of the last embedding in the vocabulary
check_emb_stats(vtok_start_id) # Check the stats of the first virtual token embedding
check_emb_stats(vtok_end_id) # Check the stats of the last virtual token embedding

i: 0
mean: -0.00018262863159179688, max: 0.1474609375, min: -0.1591796875
First values: [ 0.04614  0.02808  0.02478 -0.01636 -0.01843]

i: 151664
mean: 3.9696693420410156e-05, max: 0.035400390625, min: -0.0361328125
First values: [-0.007416  0.004974 -0.002106  0.00787  -0.00586 ]

i: 152063
mean: -0.00016260147094726562, max: 0.0972900390625, min: -0.093017578125
First values: [-0.00749   0.01172  -0.01846   0.00923  -0.012825]

i: 151669
mean: 0.0003662109375, max: 0.114013671875, min: -0.124267578125
First values: [-0.014824  0.01758  -0.02773   0.02068  -0.01784 ]

i: 159860
mean: 0.0005855560302734375, max: 0.0904541015625, min: -0.0919189453125
First values: [ 0.01361  -0.01486  -0.048    -0.013565  0.003601]



In [25]:
# Verify that the added embeddings match the scaled codebook
embed_tokens_added = getattr(model, 'model', model).language_model.embed_tokens.weight[-K:]
lm_head_added = model.lm_head.weight[-K:]

assert torch.allclose(codebook_scaled, embed_tokens_added) and torch.allclose(codebook_scaled, lm_head_added), "The added embeddings do not match the scaled codebook!"

def check_stats(x):
    print(f"mean: {x.mean().item()}, max: {x.max().item()}, min: {x.min().item()}")

print("Last scaled codebook embedding stats:")
check_stats(codebook_scaled[-1])
print()
print("Last input embedding stats:")
check_stats(embed_tokens_added[-1])
print()
print("Last output embedding stats:")
check_stats(lm_head_added[-1])

Last scaled codebook embedding stats:
mean: 0.0005855560302734375, max: 0.0904541015625, min: -0.0919189453125

Last input embedding stats:
mean: 0.0005855560302734375, max: 0.0904541015625, min: -0.0919189453125

Last output embedding stats:
mean: 0.0005855560302734375, max: 0.0904541015625, min: -0.0919189453125


In [26]:
for n, p in model.named_parameters():
    assert not p.is_meta, "Still meta tensor!"
    # print(n, p.is_meta)

In [27]:
model.config.vocab_size = len(tokenizer.tokenizer)
print("Updated model config vocab size:", model.config.vocab_size)

Updated model config vocab size: 159861


In [30]:
HUB_MODEL_ID = f"alxxtexxr/{MODEL_ID.split('/')[-1]}-with-Codebook-{int(K/1024)}K"

model.push_to_hub(HUB_MODEL_ID, max_shard_size='6GB')
tokenizer.push_to_hub(HUB_MODEL_ID)

README.md:   0%|          | 0.00/553 [00:00<?, ?B/s]

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

Saved model to https://huggingface.co/alxxtexxr/Qwen3-VL-8B-Thinking-with-Codebook-8K


Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            